# Name That LLM：可提交基线

使用词和字符 TF-IDF 加线性 SVM 识别 18 类模型。按 `prompt_id` 做五折验证，评估 Balanced Accuracy；随后用全部训练数据拟合并输出 `submission.parquet`。

在 Kaggle Notebook 中添加本比赛数据，运行全部单元格，保存版本并使用 **Submit to Competition**。不要只上传本地生成的 parquet；本比赛要求代码提交。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score, recall_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

# Set NAME_THAT_LLM_DATA_DIR before starting Jupyter to override the input.
KAGGLE_DATA = Path('/kaggle/input/competitions/name-that-llm')
REQUIRED_DATA_FILES = ('train.csv', 'test.csv')


def resolve_data_dir(explicit_dir=None):
    def missing_files(directory):
        return [name for name in REQUIRED_DATA_FILES if not (directory / name).is_file()]

    if explicit_dir is not None:
        if not str(explicit_dir).strip():
            raise ValueError('NAME_THAT_LLM_DATA_DIR cannot be empty')
        directory = Path(explicit_dir).expanduser().resolve()
        missing = missing_files(directory)
        if missing:
            raise FileNotFoundError(
                f'Explicit data directory {directory} is invalid; missing: {", ".join(missing)}. '
                'No fallback was used.'
            )
        return directory

    start = Path.cwd().resolve()
    project_root = next((
        parent for parent in (start, *start.parents)
        if (parent / 'notebooks' / '01_baseline.ipynb').is_file()
        and (parent / 'requirements.txt').is_file()
    ), None)
    candidates = [KAGGLE_DATA]
    if project_root is not None:
        candidates.extend([project_root / 'data' / 'name-that-llm', project_root / 'data'])
    for directory in candidates:
        if not missing_files(directory):
            return directory.resolve()
    details = '; '.join(
        f'{directory.resolve()}: missing {", ".join(missing_files(directory))}'
        for directory in candidates
    )
    raise FileNotFoundError(
        f'Could not find competition input files. Checked: {details}. '
        'Set NAME_THAT_LLM_DATA_DIR explicitly when outside the project.'
    )


import os
DATA_DIR = resolve_data_dir(os.environ.get('NAME_THAT_LLM_DATA_DIR'))
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
N_SPLITS = 5
print(f'Data: {DATA_DIR.resolve()}')
print(f'Output: {OUTPUT_DIR.resolve()}')

In [ ]:
import hashlib
import json

# Official names are defined only here. Lexicographic order is the score-column contract.
OFFICIAL_MODEL_LABELS = ('gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3')
class_order = tuple(sorted(OFFICIAL_MODEL_LABELS))


def _locations(frame, mask):
    positions = np.flatnonzero(np.asarray(mask))[:5]
    return [{'record': int(i) + 1,
             'id': repr(frame.iloc[i]['id']) if 'id' in frame else '<missing column>'}
            for i in positions]


def _validate_fields(frame, required, source):
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f'{source}: missing required fields {missing}')
    for field in required:
        values = frame[field]
        null = values.isna()
        if null.any():
            raise ValueError(f'{source}: field {field!r} contains null; records {_locations(frame, null)}')
        non_string = ~values.map(lambda value: isinstance(value, str)).astype(bool)
        if non_string.any():
            raise ValueError(f'{source}: field {field!r} must contain strings; records {_locations(frame, non_string)}')
        blank = values.str.strip().eq('')
        if blank.any():
            raise ValueError(f'{source}: field {field!r} is empty/whitespace; records {_locations(frame, blank)}')
    duplicate = frame['id'].duplicated(keep=False)
    if duplicate.any():
        raise ValueError(f'{source}: duplicate id; records {_locations(frame, duplicate)}')


def validate_train(frame, source='<train>', require_all_labels=True):
    _validate_fields(frame, ('id', 'prompt_id', 'text', 'model'), source)
    actual = set(frame['model'])
    unknown = sorted(actual - set(class_order))
    missing = sorted(set(class_order) - actual) if require_all_labels else []
    if unknown or missing:
        invalid = ~frame['model'].isin(class_order)
        raise ValueError(f'{source}: field model label mismatch; unknown={unknown}; '
                         f'missing={missing}; invalid records={_locations(frame, invalid)}')
    return frame


def validate_test(frame, source='<test>'):
    _validate_fields(frame, ('id', 'text'), source)
    return frame


def read_input_csv(path, kind, require_all_labels=True):
    path = Path(path).expanduser().resolve()
    if kind not in ('train', 'test'):
        raise ValueError(f'{path}: unknown input kind {kind!r}')
    try:
        # Empty fields remain empty; NA/NULL/nan are literal strings, not null markers.
        frame = pd.read_csv(path, dtype='string', keep_default_na=False,
                            na_filter=False, skip_blank_lines=False, encoding='utf-8-sig')
    except (OSError, ValueError, pd.errors.ParserError) as exc:
        raise ValueError(f'{path}: could not read {kind} CSV: {exc}') from exc
    if kind == 'train':
        return validate_train(frame, str(path), require_all_labels)
    return validate_test(frame, str(path))


def file_fingerprint(path):
    path = Path(path).resolve()
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return {'path': str(path), 'size_bytes': path.stat().st_size, 'sha256': digest.hexdigest()}


def validate_sample_submission(sample, test_frame, source='<sample_submission>'):
    # Predictions here are format examples/placeholders, not evaluated predictions.
    missing = sorted({'id', 'model'} - set(sample.columns))
    if missing:
        raise ValueError(f'{source}: missing required fields {missing}')
    _validate_fields(sample, ('id',), source)
    expected, actual = set(test_frame['id']), set(sample['id'])
    if len(sample) != len(test_frame) or actual != expected:
        raise ValueError(f'{source}: id mismatch with test; '
                         f'missing IDs={sorted(expected - actual)[:5]}; '
                         f'extra IDs={sorted(actual - expected)[:5]}; '
                         f'rows={len(sample)}, expected={len(test_frame)}')
    return sample


def load_competition_data(directory, require_all_labels=True):
    directory = Path(directory).resolve()
    train_path, test_path = directory / 'train.csv', directory / 'test.csv'
    train_frame = read_input_csv(train_path, 'train', require_all_labels)
    test_frame = read_input_csv(test_path, 'test')
    summary = {'class_order': list(class_order), 'files': {}}
    for name, frame in (('train.csv', train_frame), ('test.csv', test_frame)):
        summary['files'][name] = {
            **file_fingerprint(directory / name), 'rows': len(frame),
            'columns': frame.columns.tolist(),
            'dtypes': {column: str(dtype) for column, dtype in frame.dtypes.items()}}
    summary['files']['train.csv']['class_counts'] = {
        label: int((train_frame['model'] == label).sum()) for label in class_order}
    summary['files']['train.csv']['prompt_count'] = int(train_frame['prompt_id'].nunique())
    sample_path = directory / 'sample_submission.parquet'
    sample = None
    if sample_path.exists():
        try:
            sample = pd.read_parquet(sample_path)
        except (OSError, ValueError) as exc:
            raise ValueError(f'{sample_path}: could not read sample Parquet: {exc}') from exc
        validate_sample_submission(sample, test_frame, str(sample_path))
        summary['files'][sample_path.name] = {
            **file_fingerprint(sample_path), 'rows': len(sample),
            'columns': sample.columns.tolist(),
            'dtypes': {column: str(dtype) for column, dtype in sample.dtypes.items()},
            'id_order_matches_test': sample['id'].tolist() == test_frame['id'].tolist(),
            'placeholder_value_counts': {str(value): int(count) for value, count in
                                         sample['model'].value_counts(dropna=False).items()},
            'placeholder_null_count': int(sample['model'].isna().sum()),
            'prediction_labels_validated': False}
    else:
        summary['sample_submission_status'] = 'not provided; prediction input does not require sample'
    return train_frame, test_frame, sample, summary


train, test, sample_submission, data_summary = load_competition_data(DATA_DIR)
labels = list(class_order)
print(json.dumps(data_summary, ensure_ascii=False, indent=2))

In [ ]:
import sklearn


def generate_fold_map(frame, version='folds_v1'):
    validate_train(frame)
    settings = {'folds_v1': {'n_splits': 5, 'shuffle': False, 'random_state': None},
                'folds_v2': {'n_splits': 5, 'shuffle': True, 'random_state': 20261118}}
    if version not in settings:
        raise ValueError(f'Unknown fold version: {version}')
    params = settings[version]
    splitter = GroupKFold(**params)
    result = frame[['id', 'prompt_id']].copy()
    result['fold'] = 0
    for fold, (_, valid) in enumerate(splitter.split(frame['text'], frame['model'],
                                                    groups=frame['prompt_id']), 1):
        result.iloc[valid, result.columns.get_loc('fold')] = fold
    return result.sort_values('id').reset_index(drop=True), params


def validate_fold_map(frame, mapping):
    validate_train(frame)
    required = {'id', 'prompt_id', 'fold'}
    if set(mapping.columns) != required:
        raise ValueError('Fold map must contain exactly id, prompt_id, fold')
    _validate_fields(mapping, ('id', 'prompt_id'), 'fold map')
    missing = set(frame['id']) - set(mapping['id'])
    extra = set(mapping['id']) - set(frame['id'])
    if missing or extra:
        raise ValueError(f'Fold ID coverage mismatch: missing={sorted(missing)[:5]}, extra={sorted(extra)[:5]}')
    numeric = mapping['fold'].astype('string')
    if not numeric.isin(['1', '2', '3', '4', '5']).all():
        raise ValueError('Invalid fold: expected integer IDs 1..5')
    lookup = mapping.set_index('id').loc[frame['id']]
    if lookup['prompt_id'].tolist() != frame['prompt_id'].tolist():
        raise ValueError('Fold prompt_id mismatch with training data')
    aligned = lookup['fold'].astype(int).to_numpy()
    check = frame[['prompt_id']].copy()
    check['fold'] = aligned
    if (check.groupby('prompt_id')['fold'].nunique() != 1).any():
        raise ValueError('A prompt spans multiple folds')
    stats = []
    for fold in range(1, 6):
        valid = frame.iloc[np.flatnonzero(aligned == fold)]
        fit = frame.iloc[np.flatnonzero(aligned != fold)]
        overlap = set(fit['prompt_id']) & set(valid['prompt_id'])
        if overlap:
            raise ValueError(f'Fold {fold} has prompt leakage')
        if set(valid['model']) != set(class_order) or set(fit['model']) != set(class_order):
            raise ValueError(f'Fold {fold} does not cover all official classes')
        stats.append({'fold': fold, 'n_train': len(fit), 'n_valid': len(valid),
                      'train_prompts': int(fit['prompt_id'].nunique()),
                      'valid_prompts': int(valid['prompt_id'].nunique()),
                      'prompt_overlap': len(overlap),
                      'train_class_counts': {label: int((fit['model'] == label).sum()) for label in class_order},
                      'valid_class_counts': {label: int((valid['model'] == label).sum()) for label in class_order}})
    if sum(row['n_valid'] for row in stats) != len(frame):
        raise ValueError('Validation folds do not partition the training data')
    return aligned, stats


def save_fixed_folds(frame, train_path, directory, version='folds_v1'):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    csv_path, meta_path = directory / f'{version}.csv', directory / f'{version}_metadata.json'
    if csv_path.exists() or meta_path.exists():
        raise FileExistsError(f'Refusing to overwrite fold resource: {csv_path}, {meta_path}')
    mapping, params = generate_fold_map(frame, version)
    _, stats = validate_fold_map(frame, mapping)
    # Exclusive creation: an existing resource is never silently replaced.
    with csv_path.open('x', encoding='utf-8', newline='') as stream:
        mapping.to_csv(stream, index=False, lineterminator='\n')
    metadata = {'version': version, 'train_sha256': file_fingerprint(train_path)['sha256'],
                'fold_sha256': file_fingerprint(csv_path)['sha256'],
                'algorithm': 'sklearn.model_selection.GroupKFold', 'parameters': params,
                'sklearn_version': sklearn.__version__, 'n_samples': len(frame),
                'n_prompts': int(frame['prompt_id'].nunique()), 'fold_numbering': '1..5',
                'class_order': list(class_order), 'fold_statistics': stats,
                'generation': 'scripts/check_s03.py --generate; input in official CSV order; output sorted by string id',
                'purpose': 'primary development CV' if version == 'folds_v1' else
                           'predefined stability review; not an independent unseen test set'}
    with meta_path.open('x', encoding='utf-8', newline='') as stream:
        json.dump(metadata, stream, ensure_ascii=False, indent=2)
        stream.write('\n')
    return csv_path, meta_path


def load_fixed_folds(frame, train_path, csv_path, meta_path):
    csv_path, meta_path = Path(csv_path), Path(meta_path)
    metadata = json.loads(meta_path.read_text(encoding='utf-8'))
    expected_params = {'folds_v1': {'n_splits': 5, 'shuffle': False, 'random_state': None},
                       'folds_v2': {'n_splits': 5, 'shuffle': True, 'random_state': 20261118}}
    if (metadata['version'] not in expected_params
            or metadata['algorithm'] != 'sklearn.model_selection.GroupKFold'
            or metadata['parameters'] != expected_params[metadata['version']]
            or metadata['fold_numbering'] != '1..5'):
        raise ValueError('Fold metadata algorithm/parameters mismatch')
    if metadata['version'] != csv_path.stem:
        raise ValueError('Fold metadata version mismatch')
    if metadata['train_sha256'] != file_fingerprint(train_path)['sha256']:
        raise ValueError('Training data fingerprint mismatch')
    if metadata['fold_sha256'] != file_fingerprint(csv_path)['sha256']:
        raise ValueError('Fold file fingerprint mismatch')
    if metadata['class_order'] != list(class_order):
        raise ValueError('Fold metadata class_order mismatch')
    mapping = pd.read_csv(csv_path, dtype='string', keep_default_na=False, na_filter=False)
    aligned, stats = validate_fold_map(frame, mapping)
    if (metadata['n_samples'] != len(frame) or metadata['n_prompts'] != frame['prompt_id'].nunique()
            or metadata['fold_statistics'] != stats):
        raise ValueError('Fold metadata statistics mismatch')
    return aligned, metadata


def iter_fixed_splits(frame, train_path, csv_path, meta_path):
    aligned, _ = load_fixed_folds(frame, train_path, csv_path, meta_path)
    for fold in range(1, 6):
        yield np.flatnonzero(aligned != fold), np.flatnonzero(aligned == fold)


RANDOM_SEED = 20261118


def make_model():
    return Pipeline([
        ('features', FeatureUnion([
            ('word', TfidfVectorizer(
                analyzer='word', ngram_range=(1, 2), min_df=2,
                max_features=40_000, sublinear_tf=True, dtype=np.float32,
            )),
            ('char', TfidfVectorizer(
                analyzer='char', ngram_range=(2, 5), min_df=2,
                max_features=80_000, sublinear_tf=True, dtype=np.float32,
            )),
        ])),
        ('classifier', LinearSVC(class_weight='balanced', max_iter=3_000, random_state=RANDOM_SEED)),
    ])

model = make_model()
# Fixed local resources are required; no silent regeneration.
FOLDS_DIR = next((parent / 'experiments' / 'folds'
                  for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                  if (parent / 'experiments' / 'folds' / 'folds_v1.csv').is_file()), None)
if FOLDS_DIR is None:
    raise FileNotFoundError('Fixed folds_v1 not found; run scripts/check_s03.py --generate at project root')
fixed_splits = iter_fixed_splits(train, DATA_DIR / 'train.csv',
                                FOLDS_DIR / 'folds_v1.csv', FOLDS_DIR / 'folds_v1_metadata.json')
fold_scores = []
oof_predictions = np.empty(len(train), dtype=object)

for fold, (fit_idx, valid_idx) in enumerate(
    fixed_splits, start=1
):
    fit_prompts = set(train.iloc[fit_idx]['prompt_id'])
    valid_prompts = set(train.iloc[valid_idx]['prompt_id'])
    assert fit_prompts.isdisjoint(valid_prompts)
    fold_model = clone(model)
    fold_model.fit(train.iloc[fit_idx]['text'].astype(str), train.iloc[fit_idx]['model'])
    pred = fold_model.predict(train.iloc[valid_idx]['text'].astype(str))
    oof_predictions[valid_idx] = pred
    score = balanced_accuracy_score(train.iloc[valid_idx]['model'], pred)
    fold_scores.append({'fold': fold, 'n_valid': len(valid_idx), 'balanced_accuracy': score})
    print(f'Fold {fold}: {score:.4f} ({len(valid_idx)} rows)')

fold_results = pd.DataFrame(fold_scores)
overall_score = balanced_accuracy_score(train['model'], oof_predictions)
class_results = pd.DataFrame({
    'model': labels,
    'recall': recall_score(train['model'], oof_predictions, labels=labels, average=None, zero_division=0),
})
print(f'Mean fold score: {fold_results["balanced_accuracy"].mean():.4f}')
print(f'Out-of-fold score: {overall_score:.4f}')
display(class_results.sort_values('recall').reset_index(drop=True))

fold_results.to_csv(OUTPUT_DIR / 'cv_results.csv', index=False)
class_results.to_csv(OUTPUT_DIR / 'class_recall.csv', index=False)

In [ ]:
final_model = make_model()
final_model.fit(train['text'].astype(str), train['model'])
predictions = final_model.predict(test['text'].astype(str))
submission = pd.DataFrame({'id': test['id'], 'model': predictions})

if len(submission) != len(test):
    raise ValueError('Submission row count does not match test')
if submission['id'].isna().any() or submission['model'].isna().any():
    raise ValueError('Submission contains null values')
if submission['id'].duplicated().any() or set(submission['id']) != set(test['id']):
    raise ValueError('Submission IDs do not match test IDs exactly')
if not set(submission['model']).issubset(set(labels)):
    raise ValueError('Submission contains an invalid model label')

submission_path = OUTPUT_DIR / 'submission.parquet'
submission.to_parquet(submission_path, index=False)
check = pd.read_parquet(submission_path)
if list(check.columns) != ['id', 'model'] or len(check) != len(test):
    raise ValueError('Saved parquet file failed verification')
print(f'Saved {submission_path} ({len(submission):,} rows)')
display(submission.head())